split PDF file into chunks by hand (RecursiveCharacterTextSplitter), generate embeddings, local vector search via FAISS or qdrant (docker on in-memory)
connect day 1 and 2. make a Q&A script over my own PDF

In [2]:
from pypdf import PdfReader

In [57]:
transformers_path = 'data/transformers.pdf'

In [3]:
reader = PdfReader('data/transformers.pdf')

In [4]:
len(reader.pages)

15

In [ ]:
page = reader.pages[0]
page.extract_text()

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurren

In [26]:
all_pages_obj = reader.pages[:]
print(all_pages_obj[0].extract_text())

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exp

In [43]:
all_pages_dict = {}
for number, page in enumerate(all_pages_obj):
    all_pages_dict[number] = all_pages_obj[number].extract_text()

In [59]:
def extract_text(pdf_file_path: str) -> str:
    """A function to extract text from a PDF file."""
    reader = PdfReader(pdf_file_path)
    return "\n\n".join(doc.extract_text() or "" for doc in reader.pages)

In [63]:
extracted_pdf = extract_text(transformers_path)

In [71]:
extracted_pdf

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurren

In [64]:
len(extracted_pdf)

39628

In [ ]:
num_paragraphs = len(extracted_pdf.split("\n\n"))
num_paragraphs

15

15 \n\n, 845 \n
\n стоїть в кінці кожного ряжка. щоб "відновити" абзаци треба відрізняти звичайний перенос рядка (коли наступний рядок починається з маленької літери, продовжуючи те саме речення) та кінець абзацу/заголовок, коли рядок закінчується крапкою або іншим розділовим знаком і частіше є коротшим за звичайну довжину рядка.
якщо рядок закінчується дефісом - це перенос тексту, треба склеювати. якщо закінчується на \n та наступний рядок з маленької літери - продовження речення, \n замінити на пробіл. якщо закінчується крапкою й коротший за звичайну довжину рядка - кінець абзацу або заголовок.

In [73]:
import pymupdf4llm

In [75]:
from pathlib import Path

In [108]:
def extract_markdown(pdf_file_path: str, save_md_path: str = None) -> str:
    """A function to extract llm-ready markdown from a PDF file."""
    md_text = pymupdf4llm.to_markdown(pdf_file_path)

    if save_md_path is not None:
        filename = Path(pdf_file_path).stem
        Path(f"{save_md_path}/{filename}.md").write_text(md_text, encoding="utf-8")

    return md_text

In [80]:
extracted_markdown = extract_markdown(transformers_path, "data")


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=3/4.


In [83]:
extracted_markdown

'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. \n\n# **Attention Is All You Need** \n\n**Ashish Vaswani**<sup>_∗_</sup> **Noam Shazeer**<sup>_∗_</sup> **Niki Parmar**<sup>_∗_</sup> **Jakob Uszkoreit**<sup>_∗_</sup> Google Brain Google Brain Google Research Google Research `avaswani@google.com noam@google.com nikip@google.com usz@google.com` \n\n**Llion Jones**<sup>_∗_</sup> **Aidan N. Gomez**<sup>_∗†_</sup> **Łukasz Kaiser**<sup>_∗_</sup> Google Research University of Toronto Google Brain `llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com` \n\n**Illia Polosukhin**<sup>_∗‡_</sup> `illia.polosukhin@gmail.com` \n\n## **Abstract** \n\nThe dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an a

In [82]:
num_paragraphs = len(extracted_markdown.split("\n\n"))
num_paragraphs

200

In [84]:
import tiktoken

In [85]:
# count tokens considering ~4 char / token
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [86]:
import numpy as np

In [93]:
def analyze_paragraphs(text_to_analyze: str) -> dict:
    """A function to analyze the length of paragraphs."""
    paragraphs = text_to_analyze.split("\n\n")

    all_lengths = [
        num_tokens_from_string(p.strip(), "cl100k_base")
        for p in paragraphs if p.strip()
    ]

    filtered_lengths = [length for length in all_lengths if length > 15]

    median_length, ninety_percentile = np.percentile(filtered_lengths, [50, 90])

    result_dict = {
        "num_paragraphs_before_filtering": len(all_lengths),
        "num_paragraphs_after_filtering": len(filtered_lengths),
        "median_length": float(median_length),
        "ninety_percentile": float(ninety_percentile),
    }

    return result_dict

In [94]:
analyze_dict = analyze_paragraphs(extracted_markdown)
analyze_dict

{'num_paragraphs_before_filtering': 184,
 'num_paragraphs_after_filtering': 135,
 'median_length': 63.0,
 'ninety_percentile': 130.20000000000002}

In [95]:
import os
from dotenv import load_dotenv
import bs4
from langchain_classic import hub
from langsmith import Client
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

In [96]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT")

In [100]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

In [101]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)
md_splits = md_splitter.split_text(extracted_markdown)

In [102]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=350,
    chunk_overlap=45,
)
final_splits = text_splitter.split_documents(md_splits)

In [103]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    api_key=GEMINI_API_KEY,
)

In [104]:
vectorstore = Chroma.from_documents(
    documents=final_splits,
    embedding=embeddings,
    persist_directory="data"
)

In [105]:
retriever = vectorstore.as_retriever()

In [106]:
data = retriever._get_relevant_documents("What is transformer?", run_manager="")
print(data[0])

page_content='In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.' metadata={'Header 2': '**1 Introduction**', 'Header 1': '**Attention Is All You Need**'}


In [109]:
# post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [110]:
import json
from langchain_core.prompts import ChatPromptTemplate

In [111]:
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \nContext: {context} \nAnswer:"
)

In [112]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [113]:
# chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [114]:
# question
rag_chain.invoke("What is Transformer?")

/home/dmuman/personal/git_repos/ai-engineering-sandbox/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'The Transformer is a simple network architecture for sequence transduction that eschews recurrence and convolutions entirely. Instead, it relies solely on attention mechanisms to draw global dependencies and compute representations between input and output. This model allows for significantly more parallelization, achieves superior translation quality, and requires substantially less training time than traditional models.'